# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/eshaamirmalik-sketch/flyrank-ml-internship/blob/main/work/notebooks/w04_signal_audit.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

In [13]:
!pip -q install -U duckdb huggingface_hub

from huggingface_hub import login

login()

In [14]:
import duckdb

con = duckdb.connect()

con.execute("""
CREATE SECRET hf_token (
    TYPE HUGGINGFACE,
    PROVIDER credential_chain
);
""")

print("Hugging Face authentication configured.")

Hugging Face authentication configured.


In [15]:
from huggingface_hub import hf_hub_download

file_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    repo_type="dataset"
)

print("FILE PATH:")
print(file_path)

FILE PATH:
/root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/fact_content_daily_performance/month=2026-03/data_0.parquet


In [16]:
import duckdb
import pandas as pd

con = duckdb.connect()

df = con.execute("""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    gsc_impressions,
    gsc_clicks,
    ga4_sessions,
    gsc_avg_position,
    scroll_events
FROM read_parquet(?)
WHERE report_date = '2026-03-30'
""", [file_path]).df()

print("Rows:", len(df))
print("Columns:", df.columns.tolist())

df.describe()

Rows: 331230
Columns: ['report_date', 'client_hash_id', 'content_hash_id', 'gsc_impressions', 'gsc_clicks', 'ga4_sessions', 'gsc_avg_position', 'scroll_events']


,report_date,gsc_impressions,gsc_clicks,ga4_sessions,gsc_avg_position,scroll_events
count,331230,331230.000000,331230.000000,260530.0,125504.000000,260530.0
mean,2026-03-30 00:00:00,32.974293,0.092250,0.126634,14.350348,0.016685
min,2026-03-30 00:00:00,0.000000,0.000000,0.0,0.000000,0.0
25%,2026-03-30 00:00:00,0.000000,0.000000,0.0,3.500000,0.0
50%,2026-03-30 00:00:00,0.000000,0.000000,0.0,6.857143,0.0
75%,2026-03-30 00:00:00,6.000000,0.000000,0.0,17.631579,0.0
max,2026-03-30 00:00:00,35404.000000,225.000000,342.0,495.000000,29.0
std,NaN,186.337947,0.854259,1.167024,18.544895,0.215709


In [17]:
# W4 Section 1 — key distributions

print("=== Missing / zero checks ===")

print("\nGSC impressions:")
print(df["gsc_impressions"].describe())

print("\nGSC clicks:")
print(df["gsc_clicks"].describe())

print("\nGA4 sessions:")
print(df["ga4_sessions"].describe())

print("\nGSC average position (non-zero only):")
print(
    df.loc[df["gsc_avg_position"] > 0, "gsc_avg_position"].describe()
)

print("\nScroll events:")
print(df["scroll_events"].describe())

=== Missing / zero checks ===

GSC impressions:
count    331230.000000
mean         32.974293
std         186.337947
min           0.000000
25%           0.000000
50%           0.000000
75%           6.000000
max       35404.000000
Name: gsc_impressions, dtype: float64

GSC clicks:
count    331230.000000
mean          0.092250
std           0.854259
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max         225.000000
Name: gsc_clicks, dtype: float64

GA4 sessions:
count    260530.0
mean     0.126634
std      1.167024
min           0.0
25%           0.0
50%           0.0
75%           0.0
max         342.0
Name: ga4_sessions, dtype: Float64

GSC average position (non-zero only):
count    118991.000000
mean         15.135817
std          18.730954
min           0.009009
25%           4.000000
50%           7.272727
75%          18.831989
max         495.000000
Name: gsc_avg_position, dtype: float64

Scroll events:
count    260530.0
mean     0


### Distribution observations

The March 30 decision-point slice contains 331,230 rows. GSC impressions and clicks are highly right-skewed: the median is zero for both fields, while a small number of pages have substantially larger values.

GA4 sessions and scroll events are also sparse, with zero as the median among available rows.

For GSC average position, zero values are excluded from the non-zero summary because zero represents no ranking data rather than a valid search position. Among rows with a positive position, the median position is 7.27.

These distributions support using explicit eligibility conditions and buckets for the signal tests rather than interpreting raw values alone.**bold text**


## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

In [18]:
# W4 Signal Test #1 — CTR vs position

signal1 = df[
    (df["gsc_impressions"] > 0) &
    (df["gsc_avg_position"] > 0)
].copy()

signal1["ctr"] = (
    signal1["gsc_clicks"] / signal1["gsc_impressions"]
) * 100

signal1["position_bucket"] = pd.cut(
    signal1["gsc_avg_position"],
    bins=[0, 3, 5, 10, 20, float("inf")],
    labels=["1-3", "3-5", "5-10", "10-20", "20+"],
    include_lowest=True
)

ctr_position_table = (
    signal1
    .groupby("position_bucket", observed=True)
    .agg(
        n=("content_hash_id", "size"),
        median_ctr=("ctr", "median"),
        mean_ctr=("ctr", "mean"),
        median_impressions=("gsc_impressions", "median")
    )
    .reset_index()
)

ctr_position_table

,position_bucket,n,median_ctr,mean_ctr,median_impressions
0,1-3,20819,0.0,0.421547,20.0
1,3-5,19903,0.0,0.383179,30.0
2,5-10,32833,0.0,0.281361,19.0
3,10-20,17337,0.0,0.236059,19.0
4,20+,28099,0.0,0.134784,9.0


**Verdict: CONFIRMED**

Mean CTR decreases as average position moves from the top results toward position 20+ (0.422% to 0.135%). This supports the rule's assumption that CTR should be evaluated together with search visibility/position rather than in isolation.

The pattern is directional rather than universal because the median CTR is 0% in every bucket, reflecting the sparsity of the data.


In [19]:
# W4 Signal Test #2 — Impression volume

signal2 = df.copy()

signal2["impression_bucket"] = pd.cut(
    signal2["gsc_impressions"],
    bins=[-1, 0, 5, 20, 100, float("inf")],
    labels=["0", "1-5", "6-20", "21-100", "100+"]
)

volume_table = (
    signal2
    .groupby("impression_bucket", observed=True)
    .agg(
        n=("content_hash_id", "size"),
        mean_clicks=("gsc_clicks", "mean"),
        median_clicks=("gsc_clicks", "median")
    )
    .reset_index()
)

volume_table

,impression_bucket,n,mean_clicks,median_clicks
0,0,205726,0.000000,0.0
1,1-5,39517,0.007617,0.0
2,6-20,29847,0.023687,0.0
3,21-100,32377,0.126201,0.0
4,100+,23763,1.071498,0.0


**Verdict: CONFIRMED**

Mean clicks increase as impression volume increases, from 0.000 in the zero-impression bucket to 1.071 in the 100+ impression bucket. This supports using search volume as a prioritization signal because pages with more observed impressions have more opportunity for measurable clicks.

The relationship is directional rather than universal: the median clicks value is 0 in every bucket, reflecting the sparsity of the data.

In [20]:
# W4 Signal Test #3 — Click opportunity by impression volume

signal3 = df[
    df["gsc_impressions"] > 0
].copy()

signal3["ctr"] = (
    signal3["gsc_clicks"] / signal3["gsc_impressions"]
) * 100

signal3["impression_bucket"] = pd.cut(
    signal3["gsc_impressions"],
    bins=[0, 5, 20, 100, float("inf")],
    labels=["1-5", "6-20", "21-100", "100+"],
    include_lowest=True
)

signal3_table = (
    signal3
    .groupby("impression_bucket", observed=True)
    .agg(
        n=("content_hash_id", "size"),
        mean_ctr=("ctr", "mean"),
        median_ctr=("ctr", "median")
    )
    .reset_index()
)

signal3_table

,impression_bucket,n,mean_ctr,median_ctr
0,1-5,39517,0.385868,0.0
1,6-20,29847,0.206149,0.0
2,21-100,32377,0.252263,0.0
3,100+,23763,0.285235,0.0


**Verdict: MIXED**

Mean CTR does not move consistently with impression volume. It decreases from 0.386% in the 1–5 bucket to 0.206% in the 6–20 bucket, then increases to 0.285% in the 100+ bucket. This suggests impression volume is useful for measuring opportunity, but it should not be treated as a direct predictor of CTR.

The median CTR is 0% in every bucket, which further reflects the sparsity of the data.

## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

### Flag-linked test: low-CTR visible pages

The `low_ctr_visible_page` flag relies on the idea that pages with meaningful search visibility and relatively low CTR are worth reviewing.

Signal Test #1 directly tests this assumption by comparing CTR across average-position buckets. The observed mean CTR decreases from 0.422% for positions 1–3 to 0.135% for positions 20+, so position and CTR are clearly related in this slice.

**Verdict: CONFIRMED**

This supports using position together with CTR when identifying pages for review. The evidence is directional rather than causal, and the zero median CTR across all buckets means the rule should include an eligibility/volume threshold rather than flagging every low-CTR row.

## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

### What this means in practice

The signal audit supports using CTR and search visibility together when prioritizing pages for review. Position shows a directional relationship with CTR, while impression volume provides a useful measure of search opportunity.

The results do not justify treating any single signal as a guaranteed refresh opportunity. A practical baseline should therefore combine transparent signals, apply minimum eligibility thresholds, and expose a reason code so that the ranked queue can be reviewed.

The mixed CTR-by-volume result also suggests that volume should be treated primarily as an opportunity/priority signal rather than as evidence that a page has a CTR problem.

These findings are directional and should be used for decision support rather than causal claims about search performance.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.